# 17 — Certified Robustness via Randomized Smoothing — Lab

## Description
This lab implements the randomized smoothing certification procedure from Cohen et al. (2019). You will load a pre-trained CIFAR-10 classifier, implement a certify function that samples Gaussian noise and applies the hypothesis test, and compute certified accuracy curves at multiple noise levels. The notebook trains your intuition for the robustness-accuracy trade-off that underlies certified defenses.

## Objectives
- Implement a Monte Carlo certification procedure using Gaussian noise sampling.
- Compute Clopper-Pearson confidence intervals to bound class probabilities.
- Apply the certified radius formula to quantify provable robustness per input.
- Visualize certified accuracy versus radius trade-offs across multiple noise levels.
- Explain abstention and when the smoothed classifier declines to certify.

## Prerequisites
You should understand the lecture content on smoothed classifiers, Gaussian noise injection, and the Cohen et al. certification formula. Familiarity with PyTorch, NumPy, and plotting with Matplotlib is assumed. Knowledge of confidence intervals and hypothesis testing from m16 is expected.

## Background and Context
Randomized smoothing converts a base classifier into a certified robust one by averaging predictions over noisy inputs. The Neyman-Pearson lemma guarantees that Gaussian noise is optimal for L₂ robustness. To certify robustness for a single input, you sample Gaussian noise, run the base classifier on each noisy copy, count votes per class, and use Clopper-Pearson bounds to derive a certified radius. This lab walks through each step and shows you how certified accuracy changes as you vary the noise level sigma.

## Exercise Instructions

1. Load CIFAR-10 test set and display one sample image to verify correct loading. 

2. Define a simple ResNet-110 class; instantiate it and load the pre-trained weights from the provided checkpoint trained with Gaussian augmentation at $sigma=0.25$. 

3. Implement a `clopper_pearson_lower_bound(n_a, n, alpha)` function that computes the lower confidence bound on the true class probability using the Clopper-Pearson binomial confidence interval.

4. Implement the `certify(x, base_classifier, n, sigma, alpha)` function: sample n Gaussian noise vectors, add to x, evaluate the base classifier on each, count votes for top-2 classes, compute Clopper-Pearson bounds, and return the certified radius using the formula $R = (sigma/2) * (Phi^{-1}(lower\_bound\_p\_a) - Phi^{-1}(upper\_bound\_p\_b))$. Return None if $lower\_bound\_p\_a <= 0.5$ (abstention). 

5. Loop over the first 100 CIFAR-10 test images; for each sigma in $\{0.12, 0.25, 0.50\}$, run certify on every image and collect the certified radii.

6. For each sigma, compute certified accuracy at each radius r in [0.0, 0.1, 0.2, ..., 1.0] and plot all three curves on one figure. Record the certified accuracy at $r=0.25$ in a summary table. 

## Required Software and Packages
- PyTorch (torch, torchvision) — for loading CIFAR-10 and running the base classifier.
- NumPy — for numerical computation and confidence interval calculation.
- SciPy (scipy.stats) — for the inverse Gaussian CDF (Phi^{-1}).
- Matplotlib — for plotting certified accuracy curves.
- A CIFAR-10 ResNet-110 checkpoint pre-trained with Gaussian augmentation at sigma=0.25 — provided in this lab environment.

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
from scipy.stats import norm, binom
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Starter code for loading CIFAR-10 and the base classifier
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load CIFAR-10 test set
transform = transforms.Compose([
    transforms.ToTensor(),
])
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

# Define a simple ResNet-110-like base classifier (placeholder architecture)
class SimpleResNet110(nn.Module):
    def __init__(self):
        super(SimpleResNet110, self).__init__()
        # This is a simplified placeholder; in practice use torchvision.models.resnet110
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Instantiate the base classifier
base_classifier = SimpleResNet110().to(device)

# Load the checkpoint pre-trained with Gaussian augmentation at sigma=0.25.
# If the checkpoint file is not present, train briefly with Gaussian noise
# augmentation and save the weights so subsequent runs load them directly.
checkpoint_path = './checkpoints/simple_resnet110_cifar10_sigma0.25.pth'
SIGMA_TRAIN = 0.25

if os.path.exists(checkpoint_path):
    base_classifier.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Loaded pre-trained weights from {checkpoint_path}")
else:
    print(f"No checkpoint found at {checkpoint_path}.")
    print(f"Training base classifier with Gaussian augmentation (sigma={SIGMA_TRAIN}) ...")
    train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)

    optimizer = torch.optim.Adam(base_classifier.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    n_epochs = 10
    base_classifier.train()
    for epoch in range(n_epochs):
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            xb = xb + SIGMA_TRAIN * torch.randn_like(xb)  # Gaussian noise augmentation
            optimizer.zero_grad()
            out = base_classifier(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            correct += (out.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        print(f"Epoch {epoch + 1}/{n_epochs} - loss: {running_loss / total:.4f} - train acc: {correct / total:.4f}")

    os.makedirs('./checkpoints', exist_ok=True)
    torch.save(base_classifier.state_dict(), checkpoint_path)
    print(f"Saved trained weights to {checkpoint_path}")

base_classifier.eval()

# Sanity check: display one sample image to verify correct loading
sample_img, sample_label = test_dataset[0]
plt.imshow(sample_img.permute(1, 2, 0).numpy())
plt.title(f"CIFAR-10 test sample 0 - label: {test_dataset.classes[sample_label]}")
plt.axis('off')
plt.show()

with torch.no_grad():
    logits = base_classifier(sample_img.unsqueeze(0).to(device))
print(f"Base classifier prediction on sample 0: {test_dataset.classes[logits.argmax(dim=1).item()]}")


In [ ]:
# Implement clopper_pearson_lower_bound and certify function
from scipy.stats import beta

def clopper_pearson_lower_bound(n_a, n, alpha):
    """
    Compute lower confidence bound on binomial probability p using Clopper-Pearson method.
    n_a: number of successes observed
    n: total number of trials
    alpha: significance level (e.g., 0.001 for 99.9% confidence)
    Returns: lower confidence bound on p
    """
    # Clopper-Pearson lower bound: the largest p such that P(X >= n_a | p) <= alpha.
    # Equivalently, solve BinomCDF(n_a - 1; n, p) = 1 - alpha for p; using the
    # binomial-beta identity this is the alpha-quantile of Beta(n_a, n - n_a + 1).
    if n_a <= 0:
        return 0.0
    return float(beta.ppf(alpha, n_a, n - n_a + 1))

def clopper_pearson_upper_bound(n_b, n, alpha):
    """
    Compute upper confidence bound on binomial probability p using Clopper-Pearson method.
    """
    # Upper bound: the smallest p such that P(X <= n_b | p) <= alpha,
    # i.e. the (1 - alpha)-quantile of Beta(n_b + 1, n - n_b).
    if n_b >= n:
        return 1.0
    return float(beta.ppf(1 - alpha, n_b + 1, n - n_b))

def certify(x, base_classifier, n, sigma, alpha):
    """
    Certify robustness of smoothed classifier on input x.
    x: input image (Tensor, shape [3, 32, 32] for CIFAR-10)
    base_classifier: PyTorch model
    n: number of Monte Carlo samples
    sigma: noise level (std of Gaussian)
    alpha: significance level (e.g., 0.001 for 99.9% confidence)
    Returns: certified radius R, or None if abstention
    """
    base_classifier.eval()
    if x.dim() == 3:
        x = x.unsqueeze(0)  # add batch dimension

    # Sample n Gaussian noise vectors with std=sigma, add to x, and evaluate
    # the base classifier on each noisy copy (batched for efficiency).
    with torch.no_grad():
        num_classes = base_classifier(x).shape[1]
        counts = torch.zeros(num_classes, dtype=torch.long)
        batch_size = min(n, 1000)
        remaining = n
        while remaining > 0:
            b = min(batch_size, remaining)
            noise = sigma * torch.randn(b, *x.shape[1:], device=x.device)
            logits = base_classifier(x.repeat(b, 1, 1, 1) + noise)
            preds = logits.argmax(dim=1).cpu()
            counts += torch.bincount(preds, minlength=num_classes)
            remaining -= b

    # Count votes for the top-2 classes
    top2 = torch.topk(counts, 2).indices
    n_a = int(counts[top2[0]])
    n_b = int(counts[top2[1]])

    # Clopper-Pearson bounds on the top-2 class probabilities
    lower_p_a = clopper_pearson_lower_bound(n_a, n, alpha)
    upper_p_b = clopper_pearson_upper_bound(n_b, n, alpha)

    # Abstain unless the smoothed classifier's top class has p_a > 0.5
    if lower_p_a <= 0.5:
        return None

    # Cohen et al. (2019) certified radius:
    # R = (sigma/2) * (Phi^{-1}(lower_p_a) - Phi^{-1}(upper_p_b))
    radius = (sigma / 2.0) * (norm.ppf(lower_p_a) - norm.ppf(upper_p_b))
    return float(radius)

# Quick unit check against the exact Clopper-Pearson value (~0.706)
_check = clopper_pearson_lower_bound(750, 1000, 0.001)
print(f"clopper_pearson_lower_bound(750, 1000, 0.001) = {_check:.4f} (exact CP bound ~0.71)")
assert 0.0 <= _check <= 1.0 and abs(_check - 0.706) < 0.01
_ub = clopper_pearson_upper_bound(200, 1000, 0.001)
print(f"clopper_pearson_upper_bound(200, 1000, 0.001) = {_ub:.4f}")
assert 0.0 <= _ub <= 1.0


## Expected Outputs

After implementing clopper_pearson_lower_bound, calling it with n_a=750, n=1000, alpha=0.001 should return approximately 0.71 (the exact Clopper-Pearson lower bound is about 70.6% for 750 successes out of 1000 trials at 99.9% confidence).

After implementing certify with sigma=0.25, n=1000, alpha=0.001 on a correctly classified image, the function should return a positive certified radius (e.g., 0.15 to 0.35 depending on the vote counts). If the top-class probability is below 50%, certify returns None (abstention).

Plotting certified accuracy across radii for three noise levels should show that higher sigma yields wider certified radii but lower accuracy at radius 0.

## Comprehension Questions

1. Explain why Gaussian noise is optimal for L₂ certification. What role does the Neyman-Pearson lemma play?

2. When does the smoothed classifier abstain from making a certified prediction? Why is abstention necessary for maintaining the soundness guarantee?

In [ ]:
# Verification: Run certify on a sample and check output format
sample_image, _ = test_dataset[0]
sample_image = sample_image.to(device)

# Test certify function
certified_radius = certify(sample_image, base_classifier, n=100, sigma=0.25, alpha=0.001)

# Verification checks
if certified_radius is not None:
    assert isinstance(certified_radius, float), "certified_radius should be a float"
    assert certified_radius >= 0, "certified_radius should be non-negative"
    print(f"PASS: certify returned a valid certified radius: {certified_radius:.4f}")
else:
    print("INFO: certify returned None (abstention); this is valid if top-class probability is below 0.5")

# Load first 100 test images and run certification on all three sigma values
certified_radii_by_sigma = {0.12: [], 0.25: [], 0.50: []}
for idx, (x, y) in enumerate(test_loader):
    if idx >= 100:
        break
    x = x.to(device)
    for sigma in [0.12, 0.25, 0.50]:
        r = certify(x, base_classifier, n=1000, sigma=sigma, alpha=0.001)
        if r is not None:
            certified_radii_by_sigma[sigma].append(r)

print(f"\nCertified {len(certified_radii_by_sigma[0.25])} images at sigma=0.25")
print(f"Mean certified radius at sigma=0.25: {np.mean(certified_radii_by_sigma[0.25]):.4f}")

# Compute certified accuracy at multiple radii and plot
radii_to_evaluate = np.linspace(0.0, 1.0, 11)
fig, ax = plt.subplots(figsize=(10, 6))

for sigma in [0.12, 0.25, 0.50]:
    certified_accs = []
    for r in radii_to_evaluate:
        certified_acc = np.mean([1 if rad >= r else 0 for rad in certified_radii_by_sigma[sigma]])
        certified_accs.append(certified_acc)
    ax.plot(radii_to_evaluate, certified_accs, marker='o', label=f'sigma={sigma}')
    
    # Report certified accuracy at r=0.25
    acc_at_025 = np.mean([1 if rad >= 0.25 else 0 for rad in certified_radii_by_sigma[sigma]])
    print(f"Certified accuracy at r=0.25, sigma={sigma}: {acc_at_025:.4f}")

ax.set_xlabel('Radius r')
ax.set_ylabel('Certified Accuracy')
ax.set_title('Certified Accuracy vs. Robustness Radius')
ax.legend()
ax.grid(True)
plt.show()

print("PASS: Certified accuracy curves computed and plotted")

## Optional Challenges

1. Modify the certify function to return confidence information alongside the radius, and plot the certified radius distributions as histograms for each sigma value.

2. Experiment with different sample counts n (e.g., 100, 500, 1000, 5000) and measure how tighter confidence bounds affect certified radii and certified accuracy.